In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics

In [2]:
# ==========================================
# 1. SYNTHETIC MULTIMODAL DATA GENERATOR
# ==========================================
def generate_realistic_battery_data(num_samples=800, seq_len=30, img_size=(224, 224)):
    """
    Simulates physically coupled battery cycles:
    - 1D Telemetry: Voltage (V), Current (I), Temp (T), Internal Res (R_int)
    - 2D Thermal: Ambient base with localized Gaussian hot-spots for faulty cells
    - Targets: Correlated SOH % and Hazard Risk (0: Safe, 1: Warning, 2: Runaway)
    """
    print(f"Generating {num_samples} physically correlated multimodal samples...")

    X_telemetry = np.zeros((num_samples, seq_len, 4), dtype=np.float32)
    X_images = np.zeros((num_samples, img_size[0], img_size[1], 3), dtype=np.float32)
    y_soh = np.zeros((num_samples, 1), dtype=np.float32)
    y_risk = np.zeros((num_samples, 3), dtype=np.float32)

    for i in range(num_samples):
        # Sample operating state
        risk_class = np.random.choice([0, 1, 2], p=[0.55, 0.30, 0.15])
        
        # SOH correlated with aging cycle count
        degradation = np.random.uniform(0.0, 0.35)
        base_soh = 1.0 - degradation

        # Base nominal telemetry
        voltage = np.linspace(4.2 - degradation * 0.4, 3.6 - degradation * 0.6, seq_len)
        current = np.random.uniform(1.0, 2.5, seq_len)
        temp = np.linspace(25.0, 32.0 + degradation * 15.0, seq_len)
        r_int = np.ones(seq_len) * (0.015 + degradation * 0.04)

        # Baseline ambient image (blue/dark purple colormap)
        img = np.ones((img_size[0], img_size[1], 3), dtype=np.float32) * 0.15

        if risk_class == 1:
            # Thermal Warning: elevated temperature, voltage sag, mild hot-spot
            temp += np.linspace(0, 35.0, seq_len)
            voltage -= 0.25
            r_int += 0.03
            base_soh -= 0.08
            # Add mild hot spot in center
            cv2.circle(img, (112, 112), 40, (0.7, 0.5, 0.1), -1)

        elif risk_class == 2:
            # Critical Runaway: extreme thermal spike (>80°C), voltage collapse, bright hot-spot
            temp += np.linspace(10.0, 65.0, seq_len)
            voltage -= np.linspace(0.2, 1.2, seq_len)
            current *= 2.0
            r_int += 0.08
            base_soh -= 0.15
            # Add severe bright localized thermal runaway hot-spot
            cv2.circle(img, (112, 112), 65, (1.0, 0.2, 0.0), -1)

        # Normalize telemetry to [0, 1]
        v_norm = (voltage - 2.5) / (4.3 - 2.5)
        i_norm = current / 5.0
        t_norm = (temp - 20.0) / (100.0 - 20.0)
        r_norm = r_int / 0.15

        X_telemetry[i] = np.stack([v_norm, i_norm, t_norm, r_norm], axis=-1)
        
        # Apply Gaussian blur for natural thermal diffusion
        img = cv2.GaussianBlur(img, (21, 21), 0)
        X_images[i] = np.clip(img, 0.0, 1.0)

        y_soh[i] = np.clip(base_soh, 0.50, 1.0)
        y_risk[i, risk_class] = 1.0

    return (X_images, X_telemetry), (y_soh, y_risk)

In [3]:
def build_multimodal_model(img_shape=(224, 224, 3), seq_shape=(30, 4), num_classes=3):
    image_input = layers.Input(shape=img_shape, name="thermal_image_input")
    base_mobilenet = tf.keras.applications.MobileNetV2(
        input_shape=img_shape, include_top=False, weights="imagenet"
    )
    base_mobilenet.trainable = False
    
    spatial_features = base_mobilenet(image_input)
    spatial_pooled = layers.GlobalAveragePooling2D(name="spatial_gap")(spatial_features)
    spatial_dense = layers.Dense(128, activation="relu", name="spatial_embedding")(spatial_pooled)
    spatial_out = layers.Dropout(0.2)(spatial_dense)

    telemetry_input = layers.Input(shape=seq_shape, name="telemetry_seq_input")
    temporal_conv = layers.Conv1D(filters=32, kernel_size=3, padding="same", activation="relu")(telemetry_input)
    temporal_lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=False), name="bilstm_layer")(temporal_conv)
    temporal_dense = layers.Dense(64, activation="relu", name="temporal_embedding")(temporal_lstm)
    temporal_out = layers.Dropout(0.2)(temporal_dense)

    fused_features = layers.concatenate([spatial_out, temporal_out], name="feature_concatenation")
    attention_weights = layers.Dense(192, activation="softmax", name="fusion_attention_weights")(fused_features)
    gated_features = layers.multiply([fused_features, attention_weights], name="attention_multiplication")
    fusion_dense = layers.Dense(64, activation="relu", name="shared_representation")(gated_features)

    soh_output = layers.Dense(1, activation="linear", name="soh_output")(fusion_dense)
    risk_output = layers.Dense(num_classes, activation="softmax", name="risk_output")(fusion_dense)

    model = models.Model(
        inputs=[image_input, telemetry_input],
        outputs=[soh_output, risk_output],
        name="Multimodal_EV_Battery_Health_Net"
    )
    return model

In [4]:
# 3. PIPELINE EXECUTION & TRAINING
# ==========================================
if __name__ == "__main__":
    os.makedirs("models", exist_ok=True)
    (X_img_train, X_tel_train), (y_soh_train, y_risk_train) = generate_realistic_battery_data(num_samples=700)
    (X_img_val, X_tel_val), (y_soh_val, y_risk_val) = generate_realistic_battery_data(num_samples=150)

    model = build_multimodal_model()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss={"soh_output": losses.MeanSquaredError(), "risk_output": losses.CategoricalCrossentropy()},
        loss_weights={"soh_output": 1.0, "risk_output": 2.0},
        metrics={"soh_output": [metrics.MeanAbsoluteError(name="mae")], "risk_output": [metrics.CategoricalAccuracy(name="accuracy")]}
    )

    print("\nTraining on physically-correlated battery data...")
    model.fit(
        x={"thermal_image_input": X_img_train, "telemetry_seq_input": X_tel_train},
        y={"soh_output": y_soh_train, "risk_output": y_risk_train},
        validation_data=(
            {"thermal_image_input": X_img_val, "telemetry_seq_input": X_tel_val},
            {"soh_output": y_soh_val, "risk_output": y_risk_val}
        ),
        epochs=12,
        batch_size=16,
        verbose=1
    )

    export_path = os.path.join("models", "battery_multimodal_model.keras")
    model.save(export_path)
    print(f"\nSaved high-accuracy model to: {export_path}")

Generating 700 physically correlated multimodal samples...
Generating 150 physically correlated multimodal samples...

Training on physically-correlated battery data...
Epoch 1/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 323ms/step - loss: 1.3073 - risk_output_accuracy: 0.8757 - risk_output_loss: 0.5464 - soh_output_loss: 0.2110 - soh_output_mae: 0.3711 - val_loss: 0.2286 - val_risk_output_accuracy: 1.0000 - val_risk_output_loss: 0.1039 - val_soh_output_loss: 0.0215 - val_soh_output_mae: 0.1215
Epoch 2/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 13s 285ms/step - loss: 0.5099 - risk_output_accuracy: 0.9629 - risk_output_loss: 0.2096 - soh_output_loss: 0.0891 - soh_output_mae: 0.2028 - val_loss: 0.0474 - val_risk_output_accuracy: 1.0000 - val_risk_output_loss: 0.0084 - val_soh_output_loss: 0.0299 - val_soh_output_mae: 0.1451
Epoch 3/12
44/44 ━━━━━━━━━━━━━━━━━━━━ 13s 294ms/step - loss: 0.3313 - risk_output_accuracy: 0.9671 - risk_output_loss: 0.1382 - soh_output_loss: 0.0544 - soh_output_mae: 0.1627 - val_loss: 